# Fruit Ninja: control por nariz o dedo

**N**: nariz · **D**: punta del dedo índice · **Q** o **Esc**: salir · **R**: reiniciar después del fin de partida.

Las imágenes del juego están en `fruit_ninja/assets`. Los modelos de MediaPipe están en `P5/modelos`, compartidos con el filtro de emociones; si falta alguno se descarga la primera vez.

Se usa el mismo entorno que el filtro de emociones (ver el README).

In [ ]:
import sys
from pathlib import Path

# Funciones compartidas con el filtro de emociones (P5/comun)
RAIZ_P5 = Path.cwd().resolve()
while not (RAIZ_P5 / "comun").exists() and RAIZ_P5 != RAIZ_P5.parent:
    RAIZ_P5 = RAIZ_P5.parent
sys.path.insert(0, str(RAIZ_P5 / "comun"))

from collections import deque
from dataclasses import dataclass
import math
import random
import time

import cv2
import numpy as np

from filtro_utils import (leer_png, crear_detector_cara, crear_detector_manos, Reloj,
                          imagen_mediapipe, a_pixeles, abrir_camara, hay_que_salir,
                          geometria_cara, pegar_sticker, colocar_sticker)

CARPETA_ASSETS = RAIZ_P5 / "fruit_ninja" / "assets"

# Ajustes del juego
ANCHO_CAMARA = 640
ALTO_CAMARA = 480
DURACION = 60.0
VIDAS_INICIALES = 3
INTERVALO_FRUTAS = 0.9
INTERVALO_BOMBAS = 4.5
GRAVEDAD = 1050.0
SUAVIZADO = 0.35                 # peso del punto nuevo en la media exponencial del cursor
LARGO_ESTELA = 14
MARGEN_CORTE = 8
TAMANO_MAXIMO_SPRITE = 160
VELOCIDAD_MINIMA_CORTE = 6       # píxeles por fotograma; evita cortar con el cursor quieto

# Color de las partículas al cortar cada fruta (BGR)
COLORES_FRUTAS = {
    "paraguayo.png": (40, 40, 220),
    "manzana.png": (30, 30, 220),
    "naranja.png": (0, 140, 255),
    "sandia.png": (40, 180, 50),
}

In [2]:
def reducir_sprite(sprite, tamano_maximo):
    """Reduce los sprites grandes manteniendo su proporción y transparencia."""
    alto, ancho = sprite.shape[:2]
    escala = min(1.0, tamano_maximo / max(alto, ancho))
    if escala < 1.0:
        nuevo_tamano = (max(1, int(ancho * escala)), max(1, int(alto * escala)))
        sprite = cv2.resize(sprite, nuevo_tamano, interpolation=cv2.INTER_AREA)
    return sprite


def cargar_sprite(nombre):
    sprite = leer_png(CARPETA_ASSETS / nombre)
    if sprite is None or sprite.ndim != 3 or sprite.shape[2] != 4:
        raise RuntimeError(f"{nombre} debe ser un PNG con transparencia en {CARPETA_ASSETS}")
    return sprite


SPRITES_FRUTAS = [
    (reducir_sprite(cargar_sprite(nombre), TAMANO_MAXIMO_SPRITE), color)
    for nombre, color in COLORES_FRUTAS.items()
]
SPRITE_BOMBA = reducir_sprite(cargar_sprite("bomba.png"), TAMANO_MAXIMO_SPRITE)

# La cinta se recorta al contenido para que su tamaño corresponda solo al dibujo
SPRITE_CINTA = cargar_sprite("cinta.png")
x, y, w, h = cv2.boundingRect(cv2.findNonZero((SPRITE_CINTA[:, :, 3] > 8).astype("uint8")))
SPRITE_CINTA = reducir_sprite(SPRITE_CINTA[y:y + h, x:x + w], 1200)


@dataclass
class Fruta:
    x: float
    y: float
    vx: float
    vy: float
    radio: int
    sprite: object
    color: tuple

@dataclass
class Bomba:
    x: float
    y: float
    vx: float
    vy: float
    radio: int

@dataclass
class Particula:
    x: float
    y: float
    vx: float
    vy: float
    vida: float
    vida_max: float
    color: tuple
    radio: int


def crear_fruta(ancho, alto):
    """Fruta que sale desde abajo de la pantalla con velocidad hacia arriba."""
    sprite, color = random.choice(SPRITES_FRUTAS)
    radio = random.randint(30, 40)
    return Fruta(random.randint(radio, ancho - radio), alto + radio,
                 random.uniform(-120, 120), -random.uniform(820, 950),
                 radio, sprite, color)


def crear_bomba(ancho, alto):
    radio = random.randint(25, 33)
    return Bomba(random.randint(radio, ancho - radio), alto + radio,
                 random.uniform(-100, 100), -random.uniform(840, 960), radio)


def actualizar_objetos(objetos, dt, ancho, alto):
    """Aplica velocidad y gravedad, y elimina los objetos que salen de la pantalla."""
    for objeto in objetos:
        objeto.x += objeto.vx * dt
        objeto.y += objeto.vy * dt
        objeto.vy += GRAVEDAD * dt
    objetos[:] = [
        objeto for objeto in objetos
        if objeto.y - objeto.radio <= alto + 60
        and -objeto.radio <= objeto.x <= ancho + objeto.radio
    ]


# Sprites ya redimensionados, para no repetir cv2.resize en cada fotograma
CACHE_SPRITES = {}

def sprite_escalado(sprite, diametro):
    clave = (id(sprite), diametro)
    if clave not in CACHE_SPRITES:
        alto_img, ancho_img = sprite.shape[:2]
        factor = diametro / max(alto_img, ancho_img)
        nuevo_tamano = (max(1, int(ancho_img * factor)), max(1, int(alto_img * factor)))
        interpolacion = cv2.INTER_AREA if factor < 1 else cv2.INTER_LINEAR
        CACHE_SPRITES[clave] = cv2.resize(sprite, nuevo_tamano, interpolation=interpolacion)
    return CACHE_SPRITES[clave]


def dibujar_cinta(frame, pts):
    """Coloca la cinta en la frente. El ángulo y la dirección de la cara salen de
    geometria_cara, igual que en el filtro de emociones; el ancho, de la distancia
    entre los laterales de la cara (landmarks 127 y 356)."""
    escala, angulo, u, v = geometria_cara(pts)
    lado_izq = np.array(pts[127], dtype=np.float32)
    lado_der = np.array(pts[356], dtype=np.float32)
    ancho_cara = float(np.hypot(*(lado_der - lado_izq)))
    if ancho_cara < 12:
        return

    alto, ancho = frame.shape[:2]
    ancho_cinta = max(24, min(ancho_cara * 1.35, ancho * 0.72, 420))

    # Altura: entre la parte alta de la frente (10) y las cejas (105 y 334)
    centro_x = (lado_izq[0] + lado_der[0]) / 2
    centro_y = (pts[10][1] + (pts[105][1] + pts[334][1]) / 2) / 2

    # El PNG tiene el lazo a la derecha: se desplaza para centrar el paño en la frente
    centro = np.array([centro_x, centro_y]) + 0.11 * ancho_cinta * u
    colocar_sticker(frame, SPRITE_CINTA, centro, ancho_cinta, angulo)


def dibujar_frutas(frame, frutas):
    for fruta in frutas:
        pegar_sticker(frame, sprite_escalado(fruta.sprite, fruta.radio * 2), fruta.x, fruta.y)


def dibujar_bombas(frame, bombas):
    for bomba in bombas:
        pegar_sticker(frame, sprite_escalado(SPRITE_BOMBA, bomba.radio * 2), bomba.x, bomba.y)


def distancia_punto_segmento(punto, inicio, fin):
    """Distancia de una fruta o bomba al último tramo recorrido por el cursor."""
    px, py = punto
    ax, ay = inicio
    bx, by = fin
    vx, vy = bx - ax, by - ay
    longitud2 = vx * vx + vy * vy
    if longitud2 == 0:
        qx, qy = ax, ay
    else:
        t = max(0.0, min(1.0, ((px - ax) * vx + (py - ay) * vy) / longitud2))
        qx, qy = ax + t * vx, ay + t * vy
    return math.hypot(px - qx, py - qy)


def crear_explosion(x, y, color, cantidad=18):
    """Partículas al cortar una fruta o tocar una bomba."""
    particulas = []
    for _ in range(cantidad):
        angulo = random.uniform(0, 2 * math.pi)
        velocidad = random.uniform(90, 280)
        vida = random.uniform(0.25, 0.55)
        particulas.append(Particula(x, y, math.cos(angulo) * velocidad, math.sin(angulo) * velocidad,
                                    vida, vida, color, random.randint(3, 7)))
    return particulas


def actualizar_particulas(particulas, dt):
    for p in particulas:
        p.x += p.vx * dt
        p.y += p.vy * dt
        p.vy += GRAVEDAD * 0.45 * dt
        p.vx *= 0.985
        p.vida -= dt
    particulas[:] = [p for p in particulas if p.vida > 0]


def dibujar_particulas(frame, particulas):
    for p in particulas:
        intensidad = max(0.0, p.vida / p.vida_max)
        color = tuple(int(c * intensidad) for c in p.color)
        cv2.circle(frame, (int(p.x), int(p.y)), max(1, int(p.radio * intensidad)),
                   color, -1, cv2.LINE_AA)

In [3]:
VENTANA = "Fruit Ninja - nariz o dedo"

camara = abrir_camara(ancho=ANCHO_CAMARA, alto=ALTO_CAMARA)
detector_cara = None
detector_manos = None

try:
    detector_cara = crear_detector_cara()
    detector_manos = crear_detector_manos()
    reloj = Reloj()     # un único reloj: los timestamps no se reinician entre partidas

    modo = None
    fin_partida = False
    puntos = 0
    vidas = VIDAS_INICIALES
    frutas = []
    bombas = []
    particulas = []
    estela = deque(maxlen=LARGO_ESTELA)
    punto_filtrado = None

    inicio_partida = None
    tiempo_anterior = time.perf_counter()
    timer_frutas = 0.0
    timer_bombas = 0.0
    restante = DURACION
    mensaje = ""
    mensaje_hasta = 0.0

    while True:
        ok, frame = camara.read()
        if not ok:
            print("No se pudo leer la webcam.")
            break

        # dt se limita para que la física no dé saltos si un fotograma tarda mucho
        ahora = time.perf_counter()
        dt = min(max(ahora - tiempo_anterior, 0.0), 0.05)
        tiempo_anterior = ahora

        frame = cv2.flip(frame, 1)
        alto, ancho = frame.shape[:2]
        punto = None
        pts_cara = None

        # La cara se detecta en los dos modos: controla la nariz o coloca la cinta
        if modo is not None:
            imagen = imagen_mediapipe(frame)
            marca = reloj.ms()
            resultado_cara = detector_cara.detect_for_video(imagen, marca)
            if resultado_cara.face_landmarks:
                pts_cara = a_pixeles(resultado_cara.face_landmarks[0], ancho, alto)

            if modo == "nariz" and not fin_partida and pts_cara is not None:
                punto = pts_cara[1]
            elif modo == "dedo" and not fin_partida:
                resultado_mano = detector_manos.detect_for_video(imagen, marca)
                if resultado_mano.hand_landmarks:
                    punto = a_pixeles(resultado_mano.hand_landmarks[0], ancho, alto)[8]

        if modo is not None and not fin_partida:
            # Media exponencial para suavizar el cursor
            if punto is None:
                punto_filtrado = None
                estela.clear()
            elif punto_filtrado is None:
                punto_filtrado = punto
                estela.append(punto)
            else:
                punto_filtrado = (
                    SUAVIZADO * punto[0] + (1 - SUAVIZADO) * punto_filtrado[0],
                    SUAVIZADO * punto[1] + (1 - SUAVIZADO) * punto_filtrado[1],
                )
                estela.append((round(punto_filtrado[0]), round(punto_filtrado[1])))

            if inicio_partida is not None:
                restante = max(0.0, DURACION - (time.monotonic() - inicio_partida))
            else:
                restante = DURACION

            timer_frutas += dt
            while timer_frutas >= INTERVALO_FRUTAS:
                frutas.append(crear_fruta(ancho, alto))
                timer_frutas -= INTERVALO_FRUTAS

            timer_bombas += dt
            while timer_bombas >= INTERVALO_BOMBAS:
                bombas.append(crear_bomba(ancho, alto))
                timer_bombas -= INTERVALO_BOMBAS

            actualizar_objetos(frutas, dt, ancho, alto)
            actualizar_objetos(bombas, dt, ancho, alto)

            # Solo se corta si el cursor se mueve lo suficiente
            if (len(estela) >= 2 and math.hypot(estela[-1][0] - estela[-2][0],
                                                estela[-1][1] - estela[-2][1])
                    >= VELOCIDAD_MINIMA_CORTE):
                a, b = estela[-2], estela[-1]
                restantes = []
                for fruta in frutas:
                    if distancia_punto_segmento((fruta.x, fruta.y), a, b) <= fruta.radio + MARGEN_CORTE:
                        puntos += 10
                        particulas.extend(crear_explosion(fruta.x, fruta.y, fruta.color, 20))
                        mensaje, mensaje_hasta = "+10", time.monotonic() + 0.4
                    else:
                        restantes.append(fruta)
                frutas = restantes

                restantes = []
                for bomba in bombas:
                    if distancia_punto_segmento((bomba.x, bomba.y), a, b) <= bomba.radio + MARGEN_CORTE:
                        vidas -= 1
                        particulas.extend(crear_explosion(bomba.x, bomba.y, (30, 70, 255), 26))
                        mensaje, mensaje_hasta = "BOMBA: -1 vida", time.monotonic() + 0.6
                    else:
                        restantes.append(bomba)
                bombas = restantes

            if vidas <= 0 or restante <= 0:
                fin_partida = True
                frutas.clear()
                bombas.clear()

        elif modo is None:
            restante = DURACION

        # La cinta va primero para que frutas, bombas y estela queden por delante
        if pts_cara is not None:
            dibujar_cinta(frame, pts_cara)

        actualizar_particulas(particulas, dt)
        dibujar_frutas(frame, frutas)
        dibujar_bombas(frame, bombas)
        dibujar_particulas(frame, particulas)

        for i in range(1, len(estela)):
            grosor = max(2, int(2 + 7 * i / len(estela)))
            cv2.line(frame, estela[i - 1], estela[i], (0, 220, 120), grosor, cv2.LINE_AA)
        if estela:
            cv2.circle(frame, estela[-1], 12, (0, 255, 255), -1)
            cv2.circle(frame, estela[-1], 17, (0, 0, 0), 2)

        # Pantalla de inicio o información de la partida
        if modo is None:
            capa = frame.copy()
            cv2.rectangle(capa, (20, 120), (ancho - 20, alto - 100), (15, 15, 15), -1)
            frame = cv2.addWeighted(capa, 0.72, frame, 0.28, 0)
            cv2.putText(frame, "FRUIT NINJA", (ancho // 2 - 150, alto // 2 - 45),
                        cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 255, 255), 3, cv2.LINE_AA)
            cv2.putText(frame, "Pulsa N para usar la nariz", (ancho // 2 - 180, alto // 2 + 5),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
            cv2.putText(frame, "Pulsa D para usar el dedo indice", (ancho // 2 - 200, alto // 2 + 45),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
            cv2.putText(frame, "Q para salir", (ancho // 2 - 70, alto // 2 + 90),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (180, 180, 180), 1)
        else:
            control = "Nariz (N/D cambia)" if modo == "nariz" else "Dedo indice (N/D cambia)"
            cv2.putText(frame, f"Puntos: {puntos}  Vidas: {vidas}  Tiempo: {int(restante)}s",
                        (15, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.58, (255, 255, 255), 2)
            cv2.putText(frame, f"Control: {control}",
                        (15, 58), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (255, 255, 255), 2)

            if punto is None and not fin_partida:
                aviso = "No detecto la cara" if modo == "nariz" else "Muestra la mano a la camara"
                cv2.putText(frame, aviso, (15, 88), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (0, 0, 255), 2)
            if time.monotonic() < mensaje_hasta:
                cv2.putText(frame, mensaje, (ancho // 2 - 80, 85),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 255, 255), 3)

            if fin_partida:
                capa = frame.copy()
                cv2.rectangle(capa, (20, 150), (ancho - 20, alto - 130), (15, 15, 15), -1)
                frame = cv2.addWeighted(capa, 0.72, frame, 0.28, 0)
                cv2.putText(frame, "FIN DE PARTIDA", (ancho // 2 - 170, alto // 2 - 20),
                            cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 255, 255), 3)
                cv2.putText(frame, f"Puntuacion: {puntos} | R reinicia | Q sale",
                            (ancho // 2 - 210, alto // 2 + 30),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.55, (255, 255, 255), 2)
            else:
                cv2.putText(frame, "N: nariz | D: dedo | Q: salir", (15, alto - 18),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.55, (255, 255, 255), 2)

        cv2.imshow(VENTANA, frame)
        tecla = cv2.waitKey(1) & 0xFF
        if hay_que_salir(tecla, VENTANA):
            break

        letra = chr(tecla).lower()
        if not fin_partida and letra in ("n", "d"):
            nuevo_modo = "nariz" if letra == "n" else "dedo"
            if modo != nuevo_modo:
                modo = nuevo_modo
                punto_filtrado = None
                estela.clear()
                if inicio_partida is None:
                    inicio_partida = time.monotonic()

        if fin_partida and letra == "r":
            modo = None
            fin_partida = False
            puntos = 0
            vidas = VIDAS_INICIALES
            frutas.clear()
            bombas.clear()
            particulas.clear()
            estela.clear()
            punto_filtrado = None
            inicio_partida = None
            timer_frutas = 0.0
            timer_bombas = 0.0
            tiempo_anterior = time.perf_counter()

finally:
    camara.release()
    if detector_cara is not None:
        detector_cara.close()
    if detector_manos is not None:
        detector_manos.close()
    cv2.destroyAllWindows()
    cv2.waitKey(1)   # en algunos sistemas (macOS) hace falta para que la ventana se cierre